In [ ]:
import copy
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch import nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv


# 1. Check that the shared evaluation functions are available
if (
    "calculate_metrics" not in globals()
    or "evaluate_model" not in globals()
):
    raise RuntimeError(
        "Run Code 7 first to define calculate_metrics() "
        "and evaluate_model()."
    )


# 2. Define paths and training settings
input_folder = Path("results/step_06")
output_folder = Path("results/step_08")

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

SEED = 42
HIDDEN_CHANNELS = 16
ATTENTION_HEADS = 2
LEARNING_RATE = 0.01
EPOCHS = 200
BATCH_SIZE = 16

device = torch.device("cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_num_threads(2)

torch.use_deterministic_algorithms(
    True,
    warn_only=True
)


# 3. Load the same graph datasets used by the GCN
payload = torch.load(
    input_folder / "graph_datasets.pt",
    map_location="cpu",
    weights_only=True
)

train_graphs = [
    Data(**graph)
    for graph in payload["train"]
]

validation_graphs = [
    Data(**graph)
    for graph in payload["validation"]
]

test_graphs = [
    Data(**graph)
    for graph in payload["test"]
]

references = payload["references"]
feature_names = payload["feature_names"]

target_scale = float(payload["target_scale"])
n_parts = int(payload["n_parts"])
n_features = int(payload["n_features"])
forecast_horizon = int(payload["forecast_horizon"])


# 4. Create the data loaders
train_loader = DataLoader(
    train_graphs,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

validation_loader = DataLoader(
    validation_graphs,
    batch_size=1,
    shuffle=False
)

test_loader = DataLoader(
    test_graphs,
    batch_size=1,
    shuffle=False
)


# 5. Define the GAT architecture
class GATForecaster(nn.Module):

    def __init__(
        self,
        input_channels,
        hidden_channels,
        attention_heads,
        prediction_horizon
    ):

        super().__init__()

        self.graph_attention = GATConv(
            in_channels=input_channels,
            out_channels=hidden_channels,
            heads=attention_heads,
            concat=True,
            dropout=0.0,
            add_self_loops=True
        )

        # Concatenating two heads of 16 units produces 32 features.
        self.output_layer = nn.Linear(
            hidden_channels * attention_heads,
            prediction_horizon
        )

    def forward(self, graph):

        # Learn attention weights for neighboring historical nodes.
        embeddings = self.graph_attention(
            graph.x,
            graph.edge_index
        )

        embeddings = F.relu(
            embeddings
        )

        # Select each part at the last observed historical month.
        last_month_embeddings = embeddings[
            graph.last
        ]

        # Produce six direct forecasts per part.
        predictions = self.output_layer(
            last_month_embeddings
        )

        return predictions


model = GATForecaster(
    input_channels=n_features,
    hidden_channels=HIDDEN_CHANNELS,
    attention_heads=ATTENTION_HEADS,
    prediction_horizon=forecast_horizon
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

loss_function = nn.MSELoss()


# 6. Train the GAT and select the best validation checkpoint
history = []

best_validation_loss = float("inf")
best_state = None
best_epoch = None

for epoch in range(1, EPOCHS + 1):

    model.train()

    training_loss_sum = 0.0
    training_value_count = 0

    for graph in train_loader:

        graph = graph.to(device)

        optimizer.zero_grad()

        predictions = model(graph)

        loss = loss_function(
            predictions,
            graph.y
        )

        loss.backward()
        optimizer.step()

        training_loss_sum += (
            loss.item() * graph.y.numel()
        )

        training_value_count += (
            graph.y.numel()
        )

    training_loss = (
        training_loss_sum
        / training_value_count
    )

    validation_loss, validation_metrics = evaluate_model(
        model,
        validation_loader
    )

    history.append({
        "Epoch": epoch,
        "Training_MSE_Scaled": training_loss,
        "Validation_MSE_Scaled": validation_loss,
        "Validation_MAE": validation_metrics["MAE"],
        "Validation_RMSE": validation_metrics["RMSE"]
    })

    if validation_loss < best_validation_loss:

        best_validation_loss = validation_loss

        best_state = copy.deepcopy(
            model.state_dict()
        )

        best_epoch = epoch

    if epoch == 1 or epoch % 25 == 0:

        print(
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"Training loss: {training_loss:.5f} | "
            f"Validation loss: {validation_loss:.5f} | "
            f"Validation MAE: "
            f"{validation_metrics['MAE']:.3f}"
        )


# 7. Restore and save the best model
model.load_state_dict(
    best_state
)

model.eval()

torch.save(
    {
        "state_dict": best_state,
        "input_channels": n_features,
        "hidden_channels": HIDDEN_CHANNELS,
        "attention_heads": ATTENTION_HEADS,
        "prediction_horizon": forecast_horizon,
        "target_scale": target_scale,
        "references": references,
        "feature_names": feature_names,
        "best_epoch": best_epoch,
        "best_validation_loss": best_validation_loss,
        "seed": SEED
    },
    output_folder / "GAT_best_model.pt"
)

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    output_folder / "GAT_training_history.csv",
    index=False
)

print(
    f"\nBest validation checkpoint: epoch {best_epoch}"
)


# 8. Load the dates associated with each forecast origin
window_table = pd.read_csv(
    "results/step_05/forecast_windows.csv",
    parse_dates=["Origin_Date"]
)

origin_dates = (
    window_table
    .set_index("Origin_Index")["Origin_Date"]
    .to_dict()
)


# 9. Generate test predictions
prediction_records = []

with torch.no_grad():

    for graph in test_loader:

        graph = graph.to(device)

        predictions = model(graph)

        predictions_raw = (
            predictions.cpu().numpy()
            * target_scale
        )

        predictions_raw = np.maximum(
            predictions_raw,
            0
        )

        actual_raw = (
            graph.y_raw.cpu().numpy()
        )

        origin_index = int(
            graph.origin.item()
        )

        origin_date = pd.Timestamp(
            origin_dates[origin_index]
        )

        assert predictions_raw.shape == (
            n_parts,
            forecast_horizon
        )

        for part_index, reference in enumerate(references):

            for horizon in range(1, forecast_horizon + 1):

                target_date = (
                    origin_date
                    + pd.DateOffset(months=horizon)
                )

                prediction_records.append({
                    "Model": "GAT",
                    "Reference": reference,
                    "Origin_Date": origin_date.strftime(
                        "%Y-%m-%d"
                    ),
                    "Target_Date": target_date.strftime(
                        "%Y-%m-%d"
                    ),
                    "Horizon": horizon,
                    "Actual": float(
                        actual_raw[
                            part_index,
                            horizon - 1
                        ]
                    ),
                    "Prediction": float(
                        predictions_raw[
                            part_index,
                            horizon - 1
                        ]
                    ),
                    "Is_Synthetic": True
                })

predictions_df = pd.DataFrame(
    prediction_records
)


# 10. Verify that the test targets match the common protocol
expected_targets = pd.read_csv(
    "results/step_05/test_target_identities.csv"
)

identity_columns = [
    "Reference",
    "Origin_Date",
    "Target_Date",
    "Horizon",
    "Actual"
]

actual_identities = (
    predictions_df[identity_columns]
    .sort_values(["Reference", "Horizon"])
    .reset_index(drop=True)
)

expected_identities = (
    expected_targets[identity_columns]
    .sort_values(["Reference", "Horizon"])
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    actual_identities,
    expected_identities,
    check_dtype=False
)


# 11. Calculate and save overall test metrics
test_metrics = calculate_metrics(
    predictions_df["Actual"],
    predictions_df["Prediction"]
)

metrics_df = pd.DataFrame([
    {
        "Model": "GAT",
        **test_metrics
    }
])

metrics_df.to_csv(
    output_folder / "GAT_test_metrics.csv",
    index=False
)

predictions_df.to_csv(
    output_folder / "GAT_test_predictions.csv",
    index=False
)

print("\nGAT test metrics:")
print(
    metrics_df.round(4).to_string(index=False)
)


# 12. Calculate metrics for each forecast horizon
horizon_records = []

for horizon, group in predictions_df.groupby("Horizon"):

    horizon_records.append({
        "Model": "GAT",
        "Horizon": int(horizon),
        **calculate_metrics(
            group["Actual"],
            group["Prediction"]
        )
    })

horizon_metrics_df = pd.DataFrame(
    horizon_records
)

horizon_metrics_df.to_csv(
    output_folder / "GAT_metrics_by_horizon.csv",
    index=False
)


# 13. Plot the learning curves
fig, ax = plt.subplots(
    figsize=(8, 4)
)

ax.plot(
    history_df["Epoch"],
    history_df["Training_MSE_Scaled"],
    label="Training"
)

ax.plot(
    history_df["Epoch"],
    history_df["Validation_MSE_Scaled"],
    label="Validation"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("MSE on scaled targets")
ax.set_title("GAT learning curves — synthetic data")
ax.legend()

fig.tight_layout()

fig.savefig(
    output_folder / "GAT_learning_curves.png",
    dpi=160
)

plt.show()
plt.close(fig)


# 14. Plot actual versus predicted quantities
fig, ax = plt.subplots(
    figsize=(6, 6)
)

ax.scatter(
    predictions_df["Actual"],
    predictions_df["Prediction"],
    alpha=0.5,
    s=18
)

maximum_value = max(
    predictions_df["Actual"].max(),
    predictions_df["Prediction"].max(),
    1
)

ax.plot(
    [0, maximum_value],
    [0, maximum_value],
    color="black",
    linestyle="--",
    label="Perfect prediction"
)

ax.set_xlabel("Actual demand")
ax.set_ylabel("Predicted demand")
ax.set_title("GAT predictions")
ax.legend()

fig.tight_layout()

fig.savefig(
    output_folder / "GAT_actual_vs_predicted.png",
    dpi=160
)

plt.show()
plt.close(fig)


# 15. Save the training configuration
configuration = {
    "model": "GAT",
    "hidden_channels_per_head": HIDDEN_CHANNELS,
    "attention_heads": ATTENTION_HEADS,
    "concatenate_heads": True,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizer": "Adam",
    "loss": "MSE",
    "seed": SEED,
    "device": str(device),
    "best_epoch": best_epoch,
    "best_validation_loss": best_validation_loss,
    "target_scale": target_scale,
    "prediction_horizon": forecast_horizon,
    "feature_names": feature_names,
    "test_metrics": test_metrics
}

(
    output_folder / "GAT_configuration.json"
).write_text(
    json.dumps(configuration, indent=2),
    encoding="utf-8"
)

print(f"\nResults saved in: {output_folder}")